In [1]:
import numpy as np

J, M_REP = 20, 5       # 候选回答数量；每对比较重复次数

def simulate(theta, eps, seed):
    rng = np.random.default_rng(seed)
    data = []
    for i in range(J):
        for j in range(i + 1, J):
            p = 1.0 / (1.0 + np.exp(-(theta[i] - theta[j])))
            for _ in range(M_REP):
                w = i if rng.random() < p else j
                if rng.random() < eps:      # 标注者以概率 eps 翻转判断
                    w = j if w == i else i
                data.append((w, j if w == i else i))
    return data

def fit_bt(data):
    """BT = logistic 回归；以最后一个条目为参照（theta_J = 0）"""
    n = len(data)
    X = np.zeros((n, J - 1))
    for k, (w, l) in enumerate(data):
        if w < J - 1: X[k, w] += 1.0
        if l < J - 1: X[k, l] -= 1.0
    def grad(b):
        s = 1.0 / (1.0 + np.exp(-(X @ b)))
        return (-(1.0 - s)) @ X
    def hess(b):
        s = 1.0 / (1.0 + np.exp(-(X @ b)))
        return X.T @ ((s * (1.0 - s))[:, None] * X)
    b = np.zeros(J - 1)
    for _ in range(60):
        step = np.linalg.solve(hess(b), grad(b))
        b -= step
        if np.max(np.abs(step)) < 1e-10: break
    return np.append(b, 0.0), np.sqrt(np.diag(np.linalg.inv(hess(b))))

theta_true = np.sort(np.random.default_rng(42).uniform(-2, 2, J))
theta_true -= theta_true.mean()

theta_ref = theta_true - theta_true[-1]  # 与参照项参数化同口径的真值
print("eps   RMSE    mean_SE  corr(theta_hat, theta_ref)")
for eps in [0.0, 0.1, 0.25, 0.4]:
    res = []
    for seed in range(20):
        b, se = fit_bt(simulate(theta_true, eps, 100 + seed))
        res.append((np.sqrt(np.mean((b - theta_ref) ** 2)), se.mean(),
                    np.corrcoef(b, theta_ref)[0, 1]))
    r, s, c = np.mean(res, axis=0)
    print("%.2f  %.3f   %.3f    %.3f" % (eps, r, s, c))

eps   RMSE    mean_SE  corr(theta_hat, theta_ref)
0.00  0.313   0.374    0.980
0.10  0.643   0.334    0.963
0.25  1.201   0.303    0.899
0.40  1.671   0.288    0.655


In [2]:
import numpy as np

rng = np.random.default_rng(3)
M = 3000
q = rng.normal(0.0, 1.0, M)    # 真实质量
h = rng.normal(0.0, 1.0, M)    # "风格"维度（如冗长度），真实效用中是减分项
u = q - 0.5 * h                                # 真实效用
r = 0.9 * q + 0.5 * h                          # 奖励模型评分：误把 h 当作质量
s_ref = rng.normal(0.0, 0.5, M)                # 参考策略 logits
log_ref = s_ref - s_ref.max() - np.log(np.exp(s_ref - s_ref.max()).sum())

def policy(b):
    x = s_ref + r / b
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()

print("beta     KL(pi||ref)   E_pi[r](代理)   E_pi[q-0.5h](真实)")
rows = []
for b in [2.0, 1.0, 0.5, 0.25, 0.125, 0.0625, 0.03125, 0.015625]:
    pi = policy(b)
    kl = pi @ (np.log(np.maximum(pi, 1e-300)) - log_ref)
    rows.append((b, kl, pi @ r, pi @ u))
peak = max(rows, key=lambda t: t[3])
for b, kl, er, eq in rows:
    print("%.5f   %.3f        %.3f         %.3f%s"
          % (b, kl, er, eq, "  <- 真实效用峰值" if b == peak[0] else ""))
print()
print("参考策略的真实效用 E_ref[u] = %.3f" % (np.exp(log_ref) @ u))
print("代理奖励 E[r] 随 KL 单调上升；真实效用 E[u] 在 KL≈%.2f 处见顶后回落" % peak[1])
pi = policy(0.001)
print("beta->0 时：策略集中在 argmax(r)，该点真实效用 u = %.3f（质量 q=%.3f，风格 h=%.3f）"
      % (u[np.argmax(r)], q[np.argmax(r)], h[np.argmax(r)]))

beta     KL(pi||ref)   E_pi[r](代理)   E_pi[q-0.5h](真实)
2.00000   0.131        0.543         0.349
1.00000   0.507        1.046         0.653
0.50000   1.914        1.989         1.203
0.25000   5.262        3.173         1.591  <- 真实效用峰值
0.12500   7.629        3.620         0.825
0.06250   8.451        3.701         0.377
0.03125   8.527        3.705         0.348
0.01562   8.528        3.705         0.348

参考策略的真实效用 E_ref[u] = 0.027
代理奖励 E[r] 随 KL 单调上升；真实效用 E[u] 在 KL≈5.26 处见顶后回落
beta->0 时：策略集中在 argmax(r)，该点真实效用 u = 0.348（质量 q=2.133，风格 h=3.570）


In [3]:
import numpy as np
from scipy.optimize import minimize

rng = np.random.default_rng(7)
M = 50                                  # 回答空间大小（单提示、表格分布）
u = rng.normal(0.0, 1.0, M)             # 真实效用
s_ref = rng.normal(0.0, 0.5, M)         # 参考策略 logits

def softmax(x):
    e = np.exp(x - x.max()); return e / e.sum()

def tv(p, q):
    return 0.5 * np.abs(p - q).sum()

def make_pairs(P, eps, seed):
    r = np.random.default_rng(seed)
    pairs = []
    for _ in range(P):
        i, j = r.choice(M, size=2, replace=False)
        w = i if r.random() < 1.0 / (1.0 + np.exp(-(u[i] - u[j]))) else j
        if r.random() < eps:            # 标注者以概率 eps 翻转判断
            w = j if w == i else i
        pairs.append((w, j if w == i else i))
    return pairs

def design(pairs):
    n = len(pairs); W = np.zeros((n, M))
    W[np.arange(n), [p[0] for p in pairs]] = 1.0
    W[np.arange(n), [p[1] for p in pairs]] = -1.0
    return W

def fit_bt(pairs):                       # 两阶段之第一步：BT 极大似然
    W = design(pairs)
    obj = lambda b: np.sum(np.logaddexp(0.0, -(W @ b)))
    jac = lambda b: -(1.0 - 1.0 / (1.0 + np.exp(-(W @ b)))) @ W
    return minimize(obj, np.zeros(M), jac=jac, method="L-BFGS-B").x

def dpo_policy(pairs, beta):             # 一阶段：直接最小化 DPO 损失
    W = design(pairs)
    R = s_ref[np.array([p[0] for p in pairs])] - s_ref[np.array([p[1] for p in pairs])]
    obj = lambda th: np.sum(np.logaddexp(0.0, -(beta * (W @ th - R))))
    jac = lambda th: -beta * (1.0 - 1.0 / (1.0 + np.exp(-(beta * (W @ th - R))))) @ W
    return softmax(minimize(obj, s_ref.copy(), jac=jac, method="L-BFGS-B").x)

beta, eps = 0.5, 0.1
pi_oracle = softmax(s_ref + u / beta)    # 用真实奖励代入的 RLHF 闭式最优

pairs = make_pairs(400, eps, 0)
pi_bt = softmax(s_ref + fit_bt(pairs) / beta)   # 两阶段：BT-MLE 奖励 -> 同一个闭式解
print("TV(pi_dpo, pi_BT两阶段) = %.4f" % tv(dpo_policy(pairs, beta), pi_bt))
print("TV(pi_dpo, pi_oracle)   = %.4f" % tv(dpo_policy(pairs, beta), pi_oracle))
print()
print("P     TV(dpo,oracle)  E_dpo[r_hat]  E_oracle[r_hat]  E_dpo[u]  E_oracle[u]  E_ref[u]")
for P in [100, 400, 1600]:
    acc = []
    for seed in range(10):
        rh = fit_bt(make_pairs(P, eps, seed))
        p_d = softmax(s_ref + rh / beta)
        acc.append((tv(p_d, pi_oracle), p_d @ rh, pi_oracle @ rh, p_d @ u, pi_oracle @ u))
    v = np.mean(acc, axis=0)
    print("%4d   %.3f          %.3f         %.3f          %.3f     %.3f      %.3f"
          % (P, v[0], v[1], v[2], v[3], v[4], softmax(s_ref) @ u))

TV(pi_dpo, pi_BT两阶段) = 0.0000
TV(pi_dpo, pi_oracle)   = 0.6291

P     TV(dpo,oracle)  E_dpo[r_hat]  E_oracle[r_hat]  E_dpo[u]  E_oracle[u]  E_ref[u]
 100   0.969          84.061         18.075          0.266     0.983      -0.303
 400   0.538          2.088         1.212          0.872     0.983      -0.303
1600   0.274          0.857         0.948          0.644     0.983      -0.303


In [ ]:
# from datasets import load_dataset
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from trl import DPOConfig, DPOTrainer

# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct", torch_dtype="bfloat16")
# ref_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct", torch_dtype="bfloat16")
# tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

# # 偏好数据：每条样本含 prompt / chosen / rejected 三个字段
# ds = load_dataset("json", data_files="pref_stats.jsonl", split="train")

# cfg = DPOConfig(
#     output_dir="out-dpo",
#     beta=0.1,                     # KL 约束强度；越小更新越激进，常见 0.1 起步
#     loss_type="sigmoid",          # 原始 DPO 损失；另有 rpo_alpha 等变体
#     learning_rate=5e-7,           # 显著低于 SFT（SFT 常用 1e-5 量级）
#     lr_scheduler_type="cosine",
#     warmup_ratio=0.1,
#     per_device_train_batch_size=4,
#     gradient_accumulation_steps=8,
#     num_train_epochs=1,           # 偏好数据上一般只跑 1 个 epoch 以防过拟合
#     max_length=2048,
#     max_prompt_length=1024,
#     bf16=True,
#     logging_steps=10,
#     save_strategy="steps",
#     save_steps=200,
# )

# trainer = DPOTrainer(
#     model=model, ref_model=ref_model, args=cfg,
#     train_dataset=ds, processing_class=tokenizer,
# )
# trainer.train()

In [6]:
import numpy as np

rng = np.random.default_rng(11)

# 情形A：连续奖励，提示间难度差异大（组间 sd=3，组内 sd=1）
J, G = 500, 8
mu = rng.normal(0.0, 3.0, J)                    # 提示难度（层间异质性）
R = mu[:, None] + rng.normal(0.0, 1.0, (J, G))  # r_ij = mu_j + 噪声
adv_raw = R                                     # 不减 baseline
adv_cen = R - R.mean(axis=1, keepdims=True)     # 减组内均值
sd = R.std(axis=1, ddof=1, keepdims=True)
adv_grp = adv_cen / np.where(sd > 1e-8, sd, 1.0)  # GRPO 完整归一化
print("Var(原始奖励)    = %.3f" % adv_raw.var())
print("Var(减组内均值)  = %.3f   方差缩减 %.1f%%"
      % (adv_cen.var(), 100 * (1 - adv_cen.var() / adv_raw.var())))
print("Var(GRPO归一化)  = %.3f" % adv_grp.var())
print()

# 情形B：二值奖励下无信息组（组内全对或全错）的比例
p = rng.uniform(0.05, 0.95, 2000)               # 每组通过率
print("二值奖励下无信息组的比例：")
for G2 in [4, 8, 16]:
    print("  G=%2d: %.3f" % (G2, np.mean(p**G2 + (1 - p)**G2)))
print()

# 情形C：除以组内标准差造成的尺度扭曲（同为二值组，答错比例不同）
r1, r2 = np.array([1.,1.,1.,1.,0.,0.,0.,0.]), np.array([1.,1.,0.,0.,0.,0.,0.,0.])
adv = lambda r: (r - r.mean()) / r.std(ddof=1)
for name, r in [("4对4错组", r1), ("2对6错组", r2)]:
    print(name + ": A =", np.round(adv(r), 3))
print("组2 单样本优势幅度 / 组1 = %.2f 倍" % (np.abs(adv(r2)).max() / np.abs(adv(r1)).max()))

Var(原始奖励)    = 9.622
Var(减组内均值)  = 0.871   方差缩减 90.9%
Var(GRPO归一化)  = 0.875

二值奖励下无信息组的比例：
  G= 4: 0.343
  G= 8: 0.155
  G=16: 0.055

4对4错组: A = [ 0.935  0.935  0.935  0.935 -0.935 -0.935 -0.935 -0.935]
2对6错组: A = [ 1.62  1.62 -0.54 -0.54 -0.54 -0.54 -0.54 -0.54]
组2 单样本优势幅度 / 组1 = 1.73 倍


In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
# from peft import LoraConfig, get_peft_model
# from trl import SFTTrainer

# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", torch_dtype="bfloat16")
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

# peft_cfg = LoraConfig(
#     r=16,                                  # 秩：8–64 常用；任务越复杂取越大
#     lora_alpha=32,                         # 缩放系数，常取 2r，等价于 effective lr 的放大倍数
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM",
#     target_modules=[                       # 注意力投影是必选项
#         "q_proj", "k_proj", "v_proj", "o_proj",
#         # 数据量大时可再叠加 FFN，注意显存与遗忘风险同步上升
#         "gate_proj", "up_proj", "down_proj",
#     ],
# )
# model = get_peft_model(model, peft_cfg)
# model.print_trainable_parameters()          # 典型输出：<1% trainable

# args = TrainingArguments(
#     output_dir="out-lora",
#     per_device_train_batch_size=8,
#     gradient_accumulation_steps=4,          # 有效 batch = 32
#     learning_rate=1e-4,                     # LoRA 通常比全参微调高一个数量级
#     lr_scheduler_type="cosine",
#     warmup_ratio=0.03,
#     num_train_epochs=3,
#     bf16=True,
#     logging_steps=20,
#     save_strategy="epoch",
# )

# trainer = SFTTrainer(
#     model=model, args=args,
#     train_dataset=load_sft_jsonl("sft_stats.jsonl"),   # 字段：instruction / input / output
#     processing_class=tokenizer,
#     max_seq_length=2048,
# )
# trainer.train()
# model.save_pretrained("out-lora/adapter")   # 只产出 adapter，便于多版本切换

In [7]:
import numpy as np

d_out, d_in, R_TRUE, SIGMA2 = 32, 32, 3, 0.25   # 维度；真实增量秩；噪声方差

rng = np.random.default_rng(5)
U = rng.normal(0.0, 0.5, (d_out, R_TRUE))
V = rng.normal(0.0, 0.5, (R_TRUE, d_in))
Delta_true = U @ V                              # 秩为 3 的真实增量
W0 = rng.normal(0.0, 0.3, (d_out, d_in))
X_test = rng.normal(0.0, 1.0, (5000, d_in))
Y_test = X_test @ (W0 + Delta_true).T + rng.normal(0.0, np.sqrt(SIGMA2), (5000, d_out))

def fit_full(X, R):
    """全参：对增量做普通最小二乘"""
    return np.linalg.lstsq(X, R, rcond=None)[0].T     # d_out x d_in

def fit_lora(X, R, rank, rng2, iters=12000, lr=0.02):
    """LoRA：增量 = B@A，从 B=0 起步做梯度下降（带梯度范数裁剪）"""
    B = np.zeros((d_out, rank))
    A = rng2.normal(0.0, 0.1, (rank, d_in))
    for _ in range(iters):
        D = B @ A
        Gd = 2.0 / X.shape[0] * (X @ D.T - R).T @ X   # d_out x d_in
        Gb, Ga = Gd @ A.T, B.T @ Gd
        g = np.sqrt((Gb**2).sum() + (Ga**2).sum())
        if g > 10.0:
            Gb, Ga = Gb * 10.0 / g, Ga * 10.0 / g
        B, A = B - lr * Gb, A - lr * Ga
    return B @ A

def test_mse(D):
    return np.mean((X_test @ (W0 + D).T - Y_test) ** 2)

print("全参可训练参数 %d；LoRA r=3 参数 %d；LoRA r=32 参数 %d"
      % (d_out * d_in, R_TRUE * (d_out + d_in), 32 * (d_out + d_in)))
print("噪声下限 sigma^2 = %.2f" % SIGMA2)
print()
print("n样本   全参MSE    LoRA r=3 MSE   LoRA r=32 MSE   截断SVD(r=3) MSE")
for n in [32, 256, 4096]:
    rng2 = np.random.default_rng(n)
    X = rng2.normal(0.0, 1.0, (n, d_in))
    R = X @ Delta_true.T + rng2.normal(0.0, np.sqrt(SIGMA2), (n, d_out))
    Df = fit_full(X, R)
    P, s, Q = np.linalg.svd(Df, full_matrices=False)   # Eckart-Young 最优秩 3 截断
    Dtr = (P[:, :R_TRUE] * s[:R_TRUE]) @ Q[:R_TRUE]
    print("%5d   %8.2f   %.3f          %.3f           %.3f"
          % (n, test_mse(Df), test_mse(fit_lora(X, R, R_TRUE, rng2)),
             test_mse(fit_lora(X, R, 32, rng2)), test_mse(Dtr)))
print()
sv = np.linalg.svd(fit_full(X, R), compute_uv=False)
print("n=4096 时全参 LS 解的奇异值（前8个）：", np.round(sv[:8], 2))

全参可训练参数 1024；LoRA r=3 参数 192；LoRA r=32 参数 2048
噪声下限 sigma^2 = 0.25

n样本   全参MSE    LoRA r=3 MSE   LoRA r=32 MSE   截断SVD(r=3) MSE
   32       9.16   2.264          9.159           8.537
  256       0.29   0.255          0.285           0.255
 4096       0.25   0.250          0.252           0.250

n=4096 时全参 LS 解的奇异值（前8个）： [8.9  7.75 5.06 0.08 0.08 0.07 0.07 0.06]
